# Train ACT on the bimanual table-setting demonstrations

Trains an [ACT](https://tonyzhaozh.github.io/aloha/) policy on 134 scripted-expert
episodes (47,092 frames at 25 Hz) of two SO-101 arms setting a table in MuJoCo.
Two 256x256 camera streams (`overhead`, `front`), 12 joint positions in, 12
actuator commands out.

Sized for **one 16 GB T4**. See `notebooks/README.md` for upload and runtime steps.

### The one thing to know about this dataset

The action vector mixes units: channels 0-4 and 6-10 are arm joint **position
targets in radians**, channels 5 and 11 are gripper **torques in N.m**. Section 4
verifies that normalization keeps them separate and never pools the two scales.

## 0. Configuration

Everything tunable is here. The defaults are the T4 settings.

In [ ]:
# --- data ---------------------------------------------------------------
ARCHIVE_NAME = "bimanual-demos.tar.gz"   # as uploaded to a Kaggle Dataset
REPO_ID      = "local/bimanual_table_setting"
DATASET_DIR_NAME = "bimanual_table_setting"   # top-level dir inside the archive

# --- paths --------------------------------------------------------------
DATA_ROOT = "/kaggle/temp/lerobot"        # extracted here: scratch, not an output
OUT_DIR   = "/kaggle/working/act_bimanual"  # checkpoints here: survives the session

# --- T4 sizing ----------------------------------------------------------
# Two 256 px camera streams through two ResNet18 backbones. Batch 16 measured
# comfortably inside 16 GB; drop to 8 if you hit OOM, and see README for why
# raising it is usually not what you want.
BATCH_SIZE  = 16
NUM_WORKERS = 3          # Kaggle T4 gives 4 vCPU; leave one for the main process
CHUNK_SIZE  = 100        # ACT action horizon = 4.0 s at 25 Hz
USE_AMP     = True       # fp16 on T4; ACT's VAE needs the GradScaler that comes with it

# --- schedule -----------------------------------------------------------
TOTAL_STEPS   = 100_000  # a full ACT schedule; you will need several sessions
LOG_EVERY     = 100
CKPT_EVERY    = 2_000
KEEP_CKPTS    = 2        # /kaggle/working is capped; each checkpoint is ~0.6 GB
TIME_BUDGET_H = 8.5      # stop and checkpoint before Kaggle's 12 h cut-off
GRAD_CLIP     = 10.0     # lerobot's default for ACT
SEED          = 1000

# --- the mixed-unit action layout (see section 4) ------------------------
GRIPPER_CHANNELS = (5, 11)   # torque, N.m
ARM_CHANNELS     = tuple(i for i in range(12) if i not in GRIPPER_CHANNELS)  # radians

## 1. Install lerobot 0.4.4

Pinned to the version the dataset was written with, so `codebase_version` v3.0
is read natively. This pulls its own torch build and takes several minutes; the
CUDA check right after is there because a mismatched torch is the usual way this
goes wrong on Kaggle.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--upgrade-strategy", "only-if-needed",
                "lerobot==0.4.4"], check=True)

import torch
print("torch          :", torch.__version__)
print("cuda available :", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No CUDA. Set Settings > Accelerator to a GPU, and if it was already on, the "
    "lerobot install replaced torch with a CPU build -- restart the session and rerun.")
dev = torch.cuda.get_device_properties(0)
print(f"gpu            : {dev.name}, {dev.total_memory / 1024**3:.1f} GiB")
import lerobot
print("lerobot        :", lerobot.__version__)
assert lerobot.__version__.startswith("0.4.4"), lerobot.__version__

## 2. Extract the dataset

Finds the archive anywhere under `/kaggle/input` so the Dataset slug does not
have to be hard-coded. Extraction goes to `/kaggle/temp`, keeping the 20 GB
`/kaggle/working` quota for checkpoints. It is re-extracted after a restart,
which costs well under a minute.

In [ ]:
import pathlib, tarfile, shutil, time

root = pathlib.Path(DATA_ROOT) / DATASET_DIR_NAME
if (root / "meta" / "info.json").exists():
    print(f"already extracted at {root}")
else:
    hits = sorted(pathlib.Path("/kaggle/input").rglob(ARCHIVE_NAME))
    if not hits:
        raise SystemExit(
            f"{ARCHIVE_NAME} not found under /kaggle/input.\n"
            "Add it with + Add Input > Datasets and pick the dataset you uploaded it to.")
    archive = hits[0]
    print(f"extracting {archive} ({archive.stat().st_size / 1024**2:.0f} MB)")
    pathlib.Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with tarfile.open(archive) as tf:
        tf.extractall(DATA_ROOT)
    print(f"extracted in {time.time() - t0:.0f} s")

assert (root / "meta" / "info.json").exists(), f"no meta/info.json under {root}"
print("dataset root   :", root)
for p in sorted(root.rglob("*"))[:8]:
    print("   ", p.relative_to(root))

## 3. Load it, and verify it reads as v3.0 with no conversion

A LeRobot dataset written by an older codebase is silently migrated on load.
This asserts that does not happen here: the version is v3.0 and nothing warns
about converting or upgrading.

In [ ]:
import warnings
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.dataset_metadata import CODEBASE_VERSION

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ds_probe = LeRobotDataset(REPO_ID, root=root)
    suspicious = [str(w.message) for w in caught if any(
        t in str(w.message).lower()
        for t in ("convert", "codebase", "outdated", "deprecat", "migrat"))]

info = ds_probe.meta.info
print("codebase_version :", info["codebase_version"], f"(installed reader: {CODEBASE_VERSION})")
print("episodes         :", ds_probe.meta.total_episodes)
print("frames           :", ds_probe.meta.total_frames, "| len(ds) =", len(ds_probe))
print("fps / robot_type :", ds_probe.fps, "/", ds_probe.meta.robot_type)
print("tasks            :", ds_probe.meta.total_tasks)
print("features:")
for k, v in ds_probe.meta.features.items():
    print(f"   {k:32s} {v['dtype']:8s} {tuple(v['shape'])}")

assert info["codebase_version"] == "v3.0", info["codebase_version"]
assert not suspicious, f"dataset was converted on load: {suspicious}"
assert len(ds_probe) == ds_probe.meta.total_frames
print("\nOK: reads natively as v3.0, no conversion.")

## 4. Normalization: keep radians and newton-metres apart

`action` is not one quantity:

| channels | meaning | unit |
|---|---|---|
| 0-4, 6-10 | arm joint position targets | rad |
| 5, 11 | gripper torque, negative closes | N.m |

`observation.state` is homogeneous (all 12 are joint positions in radians), so
only the action needs watching.

LeRobot normalizes with **per-element** statistics: `stats['action']['mean']` is a
12-vector, not a scalar, and `NormalizerProcessorStep` keeps it as a tensor of
that shape. Each channel is therefore standardized against its own mean and
spread, and the two units are never pooled into a shared scale. The checks below
assert that rather than trusting it, and section 6 confirms it empirically on a
real batch.

The failure mode this guards against is any *scalar* or cross-channel
normalization: with radians spanning about +/-2.8 and torques only [-0.8, 1.0], a
single shared scale would squash the gripper channel to near-constant and the
policy would never learn to open or close.

In [ ]:
import numpy as np

stats = ds_probe.meta.stats
UNITS = ["N.m" if i in GRIPPER_CHANNELS else "rad" for i in range(12)]
names = ds_probe.meta.features["action"]["names"]

# (a) statistics must be per-element, not pooled to a scalar.
for key in ("action", "observation.state"):
    for m in ("mean", "std", "min", "max"):
        v = np.asarray(stats[key][m]).reshape(-1)
        assert v.shape == (12,), f"{key}.{m} has shape {v.shape}, expected (12,) per-channel"
print("per-channel stats: OK (12 values per statistic, nothing pooled)")

# (b) no channel is degenerate, so the normalizer eps never dominates.
std = np.asarray(stats["action"]["std"]).reshape(-1)
assert std.min() > 1e-3, f"near-constant action channel {int(std.argmin())}, std={std.min():.2e}"
print(f"smallest action std: {std.min():.4f} (channel {int(std.argmin())}) -- not degenerate")

print("\naction channels")
print(f"  {'ch':>3} {'name':22s} {'unit':>4} {'mean':>9} {'std':>8} {'min':>8} {'max':>8}")
for i in range(12):
    print(f"  {i:>3} {names[i]:22s} {UNITS[i]:>4}"
          f" {stats['action']['mean'][i]:9.4f} {stats['action']['std'][i]:8.4f}"
          f" {stats['action']['min'][i]:8.4f} {stats['action']['max'][i]:8.4f}")

# (c) the two unit groups really do live on different scales, which is exactly
#     why a shared scale would be wrong.
arm_span = max(stats["action"]["max"][i] - stats["action"]["min"][i] for i in ARM_CHANNELS)
grip_span = max(stats["action"]["max"][i] - stats["action"]["min"][i] for i in GRIPPER_CHANNELS)
print(f"\nwidest arm span {arm_span:.3f} rad vs gripper span {grip_span:.3f} N.m"
      f"  ->  ratio {arm_span / grip_span:.2f}x")

## 5. Policy, processors and dataloader

`make_policy` takes the feature shapes and the normalization statistics from the
dataset metadata. `make_pre_post_processors` builds the preprocessing pipeline —
rename, batch, move to device, normalize — which the training loop applies to
every batch before the forward pass.

The dataset is reopened with `delta_timestamps` so each sample carries a
`CHUNK_SIZE`-long action chunk plus the `action_is_pad` mask ACT's loss needs.

In [ ]:
import torch
from torch.utils.data import DataLoader
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.factory import make_policy, make_pre_post_processors

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda")

cfg = ACTConfig(
    chunk_size=CHUNK_SIZE,
    n_action_steps=CHUNK_SIZE,
    device="cuda",
)
# Left at ACT's defaults: MEAN_STD for VISUAL, STATE and ACTION. Applied with
# per-element statistics, so this is per-channel -- see section 4.
print("normalization_mapping:", {k: v.value for k, v in cfg.normalization_mapping.items()})

# Reopen with the action chunk ACT trains on.
delta_timestamps = resolve_delta_timestamps(cfg, ds_probe.meta)
print("delta_timestamps keys:", list(delta_timestamps))
dataset = LeRobotDataset(REPO_ID, root=root, delta_timestamps=delta_timestamps)

policy = make_policy(cfg=cfg, ds_meta=dataset.meta)
policy.train()
n_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params / 1e6:.1f} M")

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=cfg, pretrained_path=None, dataset_stats=dataset.meta.stats)
print("preprocessor steps:", [type(s).__name__ for s in preprocessor.steps])

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
print(f"batches per epoch: {len(loader)}  (batch {BATCH_SIZE})")

## 6. Confirm on a real batch that the units stayed apart

The assertions in section 4 were about the stored statistics. This one is about
what the model actually receives: after preprocessing, every action channel
should be standardized on its own scale — mean near 0 and spread near 1 for the
radian channels *and* the torque channels alike. If a single scale had been
shared across units, the gripper channels would come out with a spread far from
1 while the arm channels looked fine.

In [ ]:
sample = next(iter(loader))
print("raw batch:")
for k in ("observation.state", "action", "action_is_pad"):
    print(f"   {k:22s} {tuple(sample[k].shape)} {sample[k].dtype}")
for k in dataset.meta.camera_keys:
    print(f"   {k:22s} {tuple(sample[k].shape)} {sample[k].dtype}")

with torch.no_grad():
    processed = preprocessor(dict(sample))

a = processed["action"].float().reshape(-1, 12).cpu().numpy()   # (B*chunk, 12)
print("\nnormalized action, per channel")
print(f"  {'ch':>3} {'unit':>4} {'mean':>8} {'std':>8}")
for i in range(12):
    print(f"  {i:>3} {UNITS[i]:>4} {a[:, i].mean():8.3f} {a[:, i].std():8.3f}")

arm_std = float(np.mean([a[:, i].std() for i in ARM_CHANNELS]))
grip_std = float(np.mean([a[:, i].std() for i in GRIPPER_CHANNELS]))
print(f"\nmean normalized spread -- arm (rad) {arm_std:.3f}, gripper (N.m) {grip_std:.3f}")

# Both groups must land on a comparable scale. A shared-scale bug shows up here
# as a gripper spread orders of magnitude away from the arm one.
assert 0.2 < grip_std < 5.0, f"gripper channels not standardized: spread {grip_std:.3f}"
assert 0.2 < arm_std < 5.0, f"arm channels not standardized: spread {arm_std:.3f}"
assert 0.2 < grip_std / arm_std < 5.0, (
    f"gripper and arm ended on different scales ({grip_std / arm_std:.2f}x) -- "
    "units are being mixed")
print("OK: radians and newton-metres are standardized independently.")

## 7. Train, with resumable checkpoints

Checkpoints go to `/kaggle/working`, which survives a session restart; the cell
picks up from the newest one automatically, so rerunning it continues rather than
starting over. Only the last `KEEP_CKPTS` are kept because the working directory
is quota'd and each is roughly 0.6 GB.

The loop also stops itself at `TIME_BUDGET_H` and checkpoints, so the session
ends with a usable checkpoint instead of being killed mid-step.

In [ ]:
import json, re, time, pathlib, shutil

out = pathlib.Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    policy.get_optim_params(),
    lr=cfg.optimizer_lr,
    weight_decay=cfg.optimizer_weight_decay,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def checkpoints():
    return sorted(out.glob("step_*"), key=lambda p: int(p.name.split("_")[1]))


def save_checkpoint(step):
    d = out / f"step_{step:07d}"
    tmp = out / f".tmp_{step:07d}"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    # The policy and the processors in lerobot's own format, so the result loads
    # with from_pretrained without any of this notebook.
    policy.save_pretrained(tmp)
    preprocessor.save_pretrained(tmp)
    postprocessor.save_pretrained(tmp)
    torch.save({"step": step,
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict()}, tmp / "training_state.pt")
    if d.exists():
        shutil.rmtree(d)
    tmp.rename(d)                       # atomic: a killed session cannot leave a half-written ckpt
    for old in checkpoints()[:-KEEP_CKPTS]:
        shutil.rmtree(old)
    return d


start_step = 0
existing = checkpoints()
if existing:
    latest = existing[-1]
    state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)
    from lerobot.policies.act.modeling_act import ACTPolicy
    policy = ACTPolicy.from_pretrained(latest)
    policy.to(device)
    policy.train()
    optimizer = torch.optim.AdamW(policy.get_optim_params(), lr=cfg.optimizer_lr,
                                  weight_decay=cfg.optimizer_weight_decay)
    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])
    start_step = int(state["step"])
    print(f"resuming from {latest.name} at step {start_step}")
else:
    print("no checkpoint found, training from scratch")

deadline = time.time() + TIME_BUDGET_H * 3600
step = start_step
it = iter(loader)
running, seen, t_log = 0.0, 0, time.time()
print(f"training to step {TOTAL_STEPS}; logging every {LOG_EVERY}, "
      f"checkpointing every {CKPT_EVERY}")

while step < TOTAL_STEPS:
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)

    batch = preprocessor(batch)
    with torch.amp.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        loss, parts = policy.forward(batch)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    step += 1
    running += loss.item()
    seen += 1

    if step % LOG_EVERY == 0:
        dt = time.time() - t_log
        rate = LOG_EVERY / dt
        mem = torch.cuda.max_memory_allocated() / 1024**3
        eta_h = (TOTAL_STEPS - step) / rate / 3600
        extra = " ".join(f"{k}={v:.4f}" for k, v in parts.items())
        print(f"step {step:>7}/{TOTAL_STEPS}  loss {running / seen:.4f}  {extra}"
          f"  grad {float(grad_norm):.2f}  {rate:.2f} it/s  {mem:.1f} GiB"
          f"  eta {eta_h:.1f} h", flush=True)
        running, seen, t_log = 0.0, 0, time.time()

    if step % CKPT_EVERY == 0:
        d = save_checkpoint(step)
        print(f"   saved {d}", flush=True)

    if time.time() > deadline:
        d = save_checkpoint(step)
        print(f"\nstopped at the {TIME_BUDGET_H} h budget, step {step}; saved {d}")
        print("Rerun this cell in a fresh session to continue from here.")
        break
else:
    d = save_checkpoint(step)
    print(f"\nreached {TOTAL_STEPS} steps; saved {d}")

## 8. What you have at the end

Each `step_*` directory is a self-contained lerobot checkpoint. Loading it needs
nothing from this notebook:

```python
from lerobot.policies.act.modeling_act import ACTPolicy
policy = ACTPolicy.from_pretrained('/kaggle/working/act_bimanual/step_0100000')
```

Download it from the session's **Output** tab, or add `/kaggle/working` as the
notebook output and it is saved with the version.

In [ ]:
for d in checkpoints():
    size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1024**3
    print(f"{d.name}  {size:.2f} GiB  {sorted(p.name for p in d.iterdir())}")